# Telling the story: the right chart and the decision memo

Core stage, lesson C15. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** choose a chart by the comparison it must show, remove what does not help it, give one color to the point and write the takeaway into the title, then write the decision memo with the decision first and the uncertainty in plain words. The last section is the project step of Part 2: a memo on a different test.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.
EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3', 'Project step']


RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong, tol):
            print("Hint:", message)
    return False


def check_all(label, known):
    """Compare several variables with their known answers: name -> (answer, tolerance). It never stops Run all."""
    got = {name: globals().get(name) for name in known}
    if all(value is None for value in got.values()):
        RESULTS[label] = None
        print(f"{label}: give " + ", ".join(known) + " a value, then run this cell again.")
        return False
    wrong = [name for name, (want, tol) in known.items() if got[name] is None or not same(got[name], want, tol)]
    RESULTS[label] = not wrong
    if wrong:
        print(f"{label}: not yet. Check " + ", ".join(wrong) + ".")
    else:
        print(f"{label}: correct.")
    return not wrong

# One look for every chart of the lesson: small figure, no top and right frame, no legend frame, space made automatically.
BLUE, GREEN, PURPLE, GOLD, RED = "#0072b2", "#00a07a", "#9a6be0", "#ad9100", "#c0392b"  # the colors of the site's figures
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[BLUE, GREEN, PURPLE, GOLD]),
    "figure.figsize": (4.5, 2.8),
    "figure.constrained_layout.use": True,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository
GRAY = "#9a9a9a"                             # the color of everything that is not the point


def read(name, **kwargs):
    try:
        return pd.read_csv(BASE + name, **kwargs)
    except Exception:
        return pd.read_csv(LOCAL + name, **kwargs)


users = read("users.csv", parse_dates=["signup_date"])
assign = read("experiment_assignments.csv", parse_dates=["assigned_at"])
subs = read("subscriptions.csv", parse_dates=["started_on", "ended_on", "updated_at"])
events = read("events.csv", parse_dates=["event_ts"], usecols=["user_id", "event_ts", "event_name", "platform"])    # 13 MB: a few seconds

# Known answer from the dataset README: 21,841 people, 15,868 experiment rows and 3,840 subscription rows.
assert (len(users), len(assign), len(subs)) == (21841, 15868, 3840)

# The platform of a person is the platform of their signup_complete event.
signup = events[events["event_name"] == "signup_complete"].drop_duplicates("user_id").set_index("user_id")
users["platform"] = users["user_id"].map(signup["platform"])


def gap_table(control, variant):
    """Rates, the gap, its 95 percent interval and the p-value for two 0/1 columns (lesson C9). Everything is a share, not points."""
    n0, n1 = len(control), len(variant)
    c0, c1 = int(control.sum()), int(variant.sum())
    p0, p1 = c0 / n0, c1 / n1
    pooled = (c0 + c1) / (n0 + n1)
    se_null = np.sqrt(pooled * (1 - pooled) * (1 / n0 + 1 / n1))
    se = np.sqrt(p0 * (1 - p0) / n0 + p1 * (1 - p1) / n1)
    return {"n0": n0, "n1": n1, "c0": c0, "c1": c1, "rate0": p0, "rate1": p1, "gap": p1 - p0, "low": p1 - p0 - 1.96 * se, "high": p1 - p0 + 1.96 * se, "p": float(2 * stats.norm.sf(abs((p1 - p0) / se_null)))}


def gap_difference(first, second):
    """Difference of two gaps from two gap_table results, with its z and two-sided p (each interval is the gap plus or minus 1.96 standard errors)."""
    d = first["gap"] - second["gap"]
    se = np.sqrt(sum(((t["high"] - t["low"]) / 3.92) ** 2 for t in (first, second)))
    return d, d / se, float(2 * stats.norm.sf(abs(d / se)))


def people_per_group(p0, lift, alpha=0.05, power=0.80):
    """People needed in EACH group to catch a true lift of `lift` (0.02 means 2 points) from a baseline `p0` (lesson C14)."""
    p_bar = p0 + lift / 2
    return int(np.ceil(2 * p_bar * (1 - p_bar) * (stats.norm.ppf(1 - alpha / 2) + stats.norm.ppf(power)) ** 2 / lift ** 2))


pw = assign[assign["experiment"] == "paywall_v2"].copy()
pw["week"] = (pw["assigned_at"].dt.normalize() - pd.Timestamp("2026-01-12")).dt.days // 7 + 1      # week 1 is January 12 to 18
weekly = pw.groupby(["week", "variant"])["converted"].mean().unstack() * 100                       # percent who bought, by week and group
weekly["gap"] = weekly["variant"] - weekly["control"]                                              # the lift, in percentage points
print(f"Data loaded: {len(users):,} people, {len(assign):,} experiment rows, {len(weekly)} weeks of paywall_v2.")

## Unit 1: The comparison picks the chart

**Predict first.** A reader of the paywall_v2 report must see how the lift changed from week to week. Which chart gets them there faster: two bars for each week (control and variant), or one bar for each week showing the gap? And in which week was the gap biggest?

In [ ]:
print(weekly.round(1).to_string())
fig, ax = plt.subplots()
weekly[["control", "variant"]].plot.bar(ax=ax, color=[GREEN, PURPLE])        # the default: two bars for each week
ax.set_title("Conversion by week")
print(len(ax.patches))
fig2, ax2 = plt.subplots()
ax2.bar(weekly.index, weekly["gap"], color=BLUE)                              # the comparison itself: one bar for each week
ax2.set_title("Lift by week")
print(len(ax2.patches), int(weekly["gap"].idxmax()), round(float(weekly["gap"].max()), 1))
plt.close("all")

## Unit 2: Take away what does not help

**Predict first.** The chart of the gap by week carries a frame on four sides, gridlines, a y axis and no labels on the bars. If every bar gets its value written on it, which of these can go without losing anything: the frame, the gridlines, the y axis?

In [ ]:
fig, ax = plt.subplots()
bars = ax.bar(weekly.index, weekly["gap"], color=GRAY)
for side in ax.spines.values():                                              # start from a cluttered chart: a frame on all four sides and gridlines
    side.set_visible(True)
ax.grid(True)
print(sum(side.get_visible() for side in ax.spines.values()), ax.yaxis.get_gridlines()[0].get_visible(), ax.yaxis.get_visible(), len(ax.texts))
for name in ("top", "right", "left"):                                        # take away: the frame, except the line the bars stand on
    ax.spines[name].set_visible(False)
ax.grid(False)                                                               # the gridlines
ax.yaxis.set_visible(False)                                                  # the y axis: every bar carries its value
ax.bar_label(bars, labels=[f"{value:+.1f}" for value in weekly["gap"]], fontsize=9)
ax.set_title("Lift by week")
print([name for name, side in ax.spines.items() if side.get_visible()], ax.yaxis.get_gridlines()[0].get_visible(), ax.yaxis.get_visible(), len(ax.texts))
plt.close("all")

## Unit 3: One color for the point, and the takeaway as the title

**Predict first.** The chart of the gap by week is stripped. The plan will use the weeks after the first, so they are the point. Write the title as one sentence that holds two numbers and claims no more than the data show, and say how sure one week's gap is.

In [ ]:
parts = {"first": pw[pw["week"] == 1], "later": pw[pw["week"] >= 2]}
first = gap_table(parts["first"].loc[parts["first"]["variant"] == "control", "converted"], parts["first"].loc[parts["first"]["variant"] == "variant", "converted"])
later = gap_table(parts["later"].loc[parts["later"]["variant"] == "control", "converted"], parts["later"].loc[parts["later"]["variant"] == "variant", "converted"])
colors = [GRAY if week == 1 else BLUE for week in weekly.index]              # one color for the weeks the plan will use, gray for the first
fig, ax = plt.subplots()
bars = ax.bar(weekly.index, weekly["gap"], color=colors)
ax.hlines(-2.5, 1.6, 8.4, color="black", linewidth=1)                        # a bracket under weeks 2 to 8
ax.text(5, -3.6, f"weeks 2 to 8 together: {100 * later['gap']:+.1f}", ha="center", fontsize=9)
ax.set_ylim(-4.5, 8)
ax.bar_label(bars, labels=[f"{value:+.1f}" for value in weekly["gap"]], fontsize=9)
title = f"Week 1 looked higher, {100 * first['gap']:+.1f} points; weeks 2 to 8 together suggest a steady lift near {100 * later['gap']:+.1f}"
ax.set_title(title, loc="left", fontsize=9)
print(title)
print(len(set(colors)))
half = np.mean([(t["high"] - t["low"]) / 2 for t in (gap_table(g.loc[g["variant"] == "control", "converted"], g.loc[g["variant"] == "variant", "converted"]) for _, g in pw.groupby("week"))])
print(f"{100 * half:.1f}")
plt.close("all")

## Unit 4: The decision memo

**Predict first.** The guardrail's interval is -1.1 to +9.3 points, with p = 0.130. Write the one sentence that tells a manager what that means.

In [ ]:
main = gap_table(pw.loc[pw["variant"] == "control", "converted"], pw.loc[pw["variant"] == "variant", "converted"])
latest = subs.sort_values("updated_at").groupby("subscription_id").tail(1)                    # the current row of each subscription
first_sub = latest.sort_values("started_on").groupby("user_id").head(1)
buyers = pw[pw["converted"]].merge(first_sub[["user_id", "started_on", "ended_on"]], on="user_id", how="left", validate="one_to_one")
buyers["canceled_30"] = (buyers["ended_on"] - buyers["started_on"]).dt.days <= 30
cancel = gap_table(buyers.loc[buyers["variant"] == "control", "canceled_30"], buyers.loc[buyers["variant"] == "variant", "canceled_30"])
days_run = (pw["assigned_at"].max().normalize() - pw["assigned_at"].min().normalize()).days + 1
buyers_a_day = int(pw["converted"].sum()) / 2 / days_run                                      # buyers a day in one group
weeks_signup = int(np.ceil(people_per_group(0.14, 0.05) / buyers_a_day / 7)) - 8              # more weeks of signups until the guardrail can rule out a 5 point rise
weeks_follow = int(np.ceil((14 + 30) / 7))                                                    # the last buyer can buy 14 days after signing up and cancel within 30 days of starting
d_gap, z_gap, p_gap = gap_difference(first, later)
memo = (
    f"Decision: keep running the paywall_v2 test for {weeks_signup} more weeks of signups, then {weeks_follow} weeks to follow the last buyers, before shipping it.\n"
    f"Context: from January 12 to March 8, half of {len(pw):,} new signups saw a redesigned paywall. The plan: ship only if it adds 2 points and early cancellations do not rise by 5 points.\n"
    f"Tension: the paywall seems to work, but week 1 looks higher than the weeks after it, so the steady lift is probably smaller, and we cannot yet rule out more early cancellations.\n"
    f"Evidence: conversion rose from {100 * main['rate0']:.1f}% to {100 * main['rate1']:.1f}%, {100 * main['gap']:.1f} points more (95% interval {100 * main['low']:.1f} to {100 * main['high']:.1f}) (about 1 time in {round(1 / main['p'], -2):.0f} if the paywall did nothing). "
    f"The gap was {100 * first['gap']:.1f} points in week 1 and {100 * later['gap']:.1f} in weeks 2 to 8 together (the difference of the gaps has p = {p_gap:.2f}: suggestive, not proven). "
    f"Early cancellations: {100 * cancel['rate0']:.1f}% of buyers in control and {100 * cancel['rate1']:.1f}% in the variant; values from {round(-100 * cancel['low'])} fewer to {round(100 * cancel['high'])} more per 100 buyers fit the data.\n"
    f"Recommendation: keep running; ship if the cancel gap's interval then stays below 5 points. Plan for a lift near {100 * later['gap']:.0f} points, not {100 * first['gap']:.1f}."
)
print(memo)

## Finish it

### Exercise 1: Finish it

**Task:** The chart of the gap by week should point at the weeks in which the variant beat control by at least 5 points. The colors are started for you. Finish the list: one color for every week of `weekly`, in week order, blue (`BLUE`) when the gap is at least 5 points and gray (`GRAY`) otherwise.

**Expected:** `answer_1` is a list of 8 colors.

In [ ]:
gaps = weekly["gap"].tolist()                                       # given: the gap of each week, in points, in week order
point_weeks = [gap >= 5 for gap in gaps]                            # given: True for the weeks that are the point
# one color for each week: BLUE for a point week, GRAY for the rest
answer_1 = None  # replace None with the list of 8 colors

In [ ]:
_ = check("Exercise 1", "answer_1", [BLUE, BLUE, GRAY, GRAY, BLUE, GRAY, GRAY, GRAY], traps={tuple([GRAY] + [BLUE] * 7): "That is the unit 3 chart, where the weeks after the first are the point. Here every week with a gap of at least 5 points is."})

<details>
<summary>Hint</summary>

Unit 3 chose a color for each week with a list comprehension: `[BLUE if week == 1 else GRAY for week in weekly.index]`. Here the test is on the gap, and `point_weeks` already holds it.

</details>

<details>
<summary>Half-solution</summary>

```python
answer_1 = [BLUE if ... else GRAY for is_point in point_weeks]
```

</details>

<details>
<summary>Solution</summary>

```python
answer_1 = [BLUE if is_point else GRAY for is_point in point_weeks]
```

Weeks 1, 2 and 5 have a gap of 5 points or more (6.6, 5.5 and 5.1), so three bars are blue. A chart with three blue bars tells the reader that three weeks matter, and which weeks they are; the colors must follow the claim the title makes.

</details>

## On your own

### Exercise 2: On your own

**Task:** The memo says that values from 1 fewer to 9 more cancellations per 100 buyers fit the data for the cancel guardrail of paywall_v2. A reader asks for it in people: how many fewer and how many more cancellations per 1,000 buyers in the variant? Before you start, say in one sentence what the result looks like: two whole numbers, one for each end of the 95 percent interval, the first below zero. Start from the table `cancel` of unit 4 (`low` and `high` are shares) and set `answer_2` to the list [low end, high end] per 1,000 buyers, each rounded to a whole number. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of two whole numbers.

In [ ]:
_ = check("Exercise 2", "answer_2", [-11, 93], traps={(-1, 9): "Those are per 100 buyers. The question is per 1,000.", (11, 93): "The low end is below zero: a fall in the cancel rate. Keep its sign."})

<details>
<summary>Hint</summary>

`cancel["low"]` and `cancel["high"]` are shares of buyers: 0.01 means 10 in 1,000. Multiply by 1,000 and round.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"A small negative number and a number in the nineties."*

```python
answer_2 = [round(1000 * cancel["low"]), ...]
```

</details>

<details>
<summary>Solution</summary>

```python
answer_2 = [round(1000 * cancel["low"]), round(1000 * cancel["high"])]
```

The interval runs from 11 fewer to 93 more cancellations per 1,000 buyers. A reader pictures 1,000 people more easily than a share of 0.093.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague writes the guardrail line of the memo with the rise in early cancellations in "points", and the memo reads "29.1 points more of the buyers canceled". Fix the line so that `answer_3` is the rise in percentage points, rounded to 1 decimal.

**Expected:** `answer_3` is a number.

In [ ]:
rise_points = 100 * (cancel["rate1"] - cancel["rate0"]) / cancel["rate0"]  # colleague: "the rise, in points"
answer_3 = round(float(rise_points), 1)

In [ ]:
_ = check("Exercise 3", "answer_3", 4.1, traps={29.1: "That is the change relative to the control rate, in percent. Points are the plain difference of the two rates: 18.1% minus 14.0%."})

<details>
<summary>Hint</summary>

Print `cancel["rate0"]` and `cancel["rate1"]`. Points are what you get by subtracting two percentages.

</details>

<details>
<summary>Half-solution</summary>

```python
rise_points = 100 * (cancel["rate1"] - ...)
```

</details>

<details>
<summary>Solution</summary>

```python
rise_points = 100 * (cancel["rate1"] - cancel["rate0"])
answer_3 = round(float(rise_points), 1)
```

14.0% to 18.1% is 4.1 points. The colleague divided by the control rate as well, which gives the relative change, 29.1 percent: 4.1 is 29.1 percent of 14.0. A manager who reads "29.1 points" would expect about 43 in 100 buyers to cancel instead of 14.

</details>

## Project step: a decision memo

**The brief.** The head of product asks about `signup_flow_v2`, the test of a new signup screen that ran from May 4 to June 7: should it ship? The split of the whole test failed (lesson C14), so look at the part of the test that did not break: the people who do not use Android. In the last section of the notebook, name these variables, rounded as shown:

- `srm_p_other`, the p-value of the chi-square test of the group counts against a 50/50 split among people who do not use Android (web or ios), rounded to 4 decimals.
- `gap_other`, variant minus control among those people, in points, rounded to 2 decimals.
- `ci_other`, its 95 percent interval in points, as a list [low, high] rounded to 2 decimals.
- `n_rerun`, the people needed in each group by a rerun that catches a true lift of 2 points at a baseline of 0.095, 80 percent power and the bar 0.05.
- `weeks_rerun`, the whole weeks from the start of the rerun to its readout: the weeks of signups at the average daily enrollment of the test (all of `sf`, 35 days), plus 2 weeks for the last signup to have 14 days to buy.

Then write `memo` with an f-string, so that every number in it comes from a variable. The check cell compares the variables with known answers; you read the memo.

In [ ]:
# Project step: write your code here, then run the check cell below.

In [ ]:
_ = check_all("Project step", {"srm_p_other": (0.1491, 5e-5), "gap_other": (0.9, 0.005), "ci_other": ([-1.18, 2.98], 0.005), "n_rerun": (3688, 0), "weeks_rerun": (9, 0)})

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")